# Alignment & RLHF — A Toy End-to-End Loop

**Category:** Alignment & RLHF · PPO, RLHF-Summarize, InstructGPT, Personal Alignment, Positive Alignment.

**What this notebook does.** Implements the full RLHF loop at miniature scale: a synthetic preference dataset, a reward model trained on those preferences, and a policy-gradient loop (the *spirit* of PPO without the full machinery) that optimizes a tiny policy against the reward model. Final cells visualize reward-hacking.

**Stays on CPU.** Everything is a tabular bandit-style problem so we can see every gradient.

## 1. The task — a bandit with a hidden 'politeness' axis

Each 'response' is a 4-dim vector. The hidden ground-truth says polite responses are those with positive scores on dims 0 and 1. The model never sees that rule; it only sees binary preferences between pairs.

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(7); np.random.seed(7)

def true_politeness(r): return r[..., 0] + r[..., 1]  # the hidden rubric

def sample_response():
    return torch.randn(4)

# Generate a preference dataset: pairs (a, b) labeled 1 if a more polite than b
N = 2000
A = torch.stack([sample_response() for _ in range(N)])
B = torch.stack([sample_response() for _ in range(N)])
labels = (true_politeness(A) > true_politeness(B)).float()
print('A shape:', A.shape, '| labels mean:', labels.mean().item())


## 2. Reward model

Bradley–Terry preference loss: $P(a \succ b) = \sigma(R(a) - R(b))$.

In [ ]:
class RewardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

rm = RewardModel()
opt = torch.optim.Adam(rm.parameters(), lr=1e-2)
rm_losses = []
for step in range(300):
    ra, rb = rm(A), rm(B)
    logits = ra - rb
    loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
    opt.zero_grad(); loss.backward(); opt.step()
    rm_losses.append(loss.item())
print('final RM loss:', round(rm_losses[-1], 4))


In [ ]:
# Sanity check: RM predictions vs true rubric on a held-out sample
with torch.no_grad():
    test = torch.randn(500, 4)
    pred = rm(test).numpy()
    truth = true_politeness(test).numpy()
plt.figure(figsize=(4,4))
plt.scatter(truth, pred, alpha=0.4, s=10)
plt.xlabel('true politeness'); plt.ylabel('RM prediction')
plt.title('Reward model vs ground-truth rubric')
plt.tight_layout(); plt.show()


## 3. Policy — a Gaussian policy over the 4-dim response space

Optimize the policy with REINFORCE against the reward model. (PPO adds a clipped surrogate; the conceptual loop is identical for a one-step bandit.)

In [ ]:
class GaussianPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.mu = nn.Parameter(torch.zeros(4))
        self.log_sigma = nn.Parameter(torch.zeros(4))
    def sample(self, n):
        sigma = self.log_sigma.exp()
        z = torch.randn(n, 4)
        x = self.mu + z * sigma
        logp = -0.5 * (((x - self.mu)/sigma)**2 + 2*self.log_sigma + np.log(2*np.pi))
        return x, logp.sum(-1)

pol = GaussianPolicy()
popt = torch.optim.Adam(pol.parameters(), lr=5e-2)
rew_history, true_rew_history = [], []
for step in range(200):
    x, logp = pol.sample(64)
    with torch.no_grad():
        r = rm(x)
    advantage = r - r.mean()
    loss = -(logp * advantage.detach()).mean()
    popt.zero_grad(); loss.backward(); popt.step()
    rew_history.append(r.mean().item())
    true_rew_history.append(true_politeness(x).mean().item())
print('policy mu (should drift positive on dims 0,1):', pol.mu.detach().numpy().round(2))


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,3))
ax.plot(rew_history, label='RM reward')
ax.plot(true_rew_history, label='True rubric')
ax.legend(); ax.set_xlabel('step'); ax.set_ylabel('mean reward')
ax.set_title('Reward over training — and the gap is reward hacking')
plt.tight_layout(); plt.show()


## My take

Watch the gap between the two lines on the chart above. The RM reward keeps climbing — the policy is *very good* at scoring high on the reward model. But the true-rubric reward saturates and may even fall. That gap is reward hacking, and it's the single failure mode that makes RLHF operationally hard.

The mitigations that actually work in production aren't algorithmic — they're process:

1. **Hold out a real human eval set** the policy is never optimized against.
2. **Periodically refresh the reward model** with new human preferences sampled from the policy's current distribution. Distribution drift kills RMs faster than anything else.
3. **KL-regularize against the base model** (the missing PPO piece). Don't let the policy wander too far from the language model it started as.

If a team tells me they're 'doing RLHF' but doesn't have a written answer for points 1–3, they're not doing RLHF — they're overfitting to a proxy.